# Worked loan traces

One Stage 1, Stage 2 and Stage 3 loan is traced from monthly terms to final provision.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

In [2]:
summary = query('select * from worked_trace_summary order by stage'); summary

,loan_id,stage,loan_ecl,gross_exposure,current_pd_12m,origination_pd_12m,current_lifetime_pd,origination_lifetime_pd,current_ltv,current_dpd,remaining_months_to_legal_maturity,property_state,origination_date,primary_stage_reason,coverage_ratio,trace_explanation
0,RM002539,1,34.7096,"363,199.1000",0.0035,0.0015,0.0285,0.0154,62.7558,0.0000,234.0000,CA,2015-09-01,No SICR trigger,0.0001,Sum monthly MPD x LGD x EAD x DF through month...
1,RM001506,2,"7,409.9148","230,184.1000",0.3609,0.0011,0.8293,0.0127,39.0000,0.0000,390.0000,CA,2015-07-01,Prior default history,0.0322,Sum monthly MPD x LGD x EAD x DF through remai...
2,RM002463,3,"12,983.6638","205,696.7900",NaN,0.0007,NaN,0.0076,48.0000,210.0000,234.0000,OK,2015-09-01,Current default/credit-impaired,0.0631,Gross exposure less scenario-specific discount...


In [3]:
monthly = query('select * from worked_trace_monthly')
monthly.groupby(['loan_id','stage','scenario','scenario_weight'], as_index=False).period_ecl.sum()

,loan_id,stage,scenario,scenario_weight,period_ecl
0,RM001506,2,Base,0.5152,"7,384.3776"
1,RM001506,2,Downside,0.1672,"7,611.0877"
2,RM001506,2,Upside,0.3176,"7,345.4070"
3,RM002463,3,Base,0.5152,"12,983.6638"
4,RM002463,3,Downside,0.1672,"12,983.6638"
5,RM002463,3,Upside,0.3176,"12,983.6638"
6,RM002539,1,Base,0.5152,34.0622
7,RM002539,1,Downside,0.1672,41.7781
8,RM002539,1,Upside,0.3176,32.0378


In [4]:
reconciliation = (monthly.groupby(['loan_id','stage','scenario','scenario_weight'], as_index=False).period_ecl.sum()
.assign(weighted_ecl=lambda x: x.period_ecl*x.scenario_weight)
.groupby(['loan_id','stage'], as_index=False).weighted_ecl.sum()
.merge(summary[['loan_id','loan_ecl']], on='loan_id'))
reconciliation['difference'] = reconciliation.weighted_ecl-reconciliation.loan_ecl
reconciliation

,loan_id,stage,weighted_ecl,loan_ecl,difference
0,RM001506,2,"7,409.9148","7,409.9148",0.0000
1,RM002463,3,"12,983.6638","12,983.6638",0.0000
2,RM002539,1,34.7096,34.7096,0.0000


In [5]:
stage1_id = summary.loc[summary.stage.eq(1),'loan_id'].iloc[0]
monthly[(monthly.loan_id.eq(stage1_id)) & (monthly.scenario.eq('Base'))][
['future_month','marginal_pd','lgd','ead','discount_factor','period_ecl']].head(12)

,future_month,marginal_pd,lgd,ead,discount_factor,period_ecl
1170,1,0.0003,0.0283,"362,897.3733",0.9964,3.1447
1171,2,0.0003,0.0283,"361,906.1802",0.9927,3.0881
1172,3,0.0003,0.0282,"360,911.3735",0.9891,3.0316
1173,4,0.0003,0.0281,"359,912.9398",0.9855,2.9754
1174,5,0.0003,0.0280,"358,910.8660",0.9820,2.9194
1175,6,0.0003,0.0279,"357,905.1388",0.9784,2.8638
1176,7,0.0003,0.0278,"356,895.7449",0.9748,2.8086
1177,8,0.0003,0.0277,"355,882.6710",0.9713,2.7538
1178,9,0.0003,0.0276,"354,865.9035",0.9678,2.6995
1179,10,0.0003,0.0275,"353,845.4291",0.9643,2.6456


In [6]:
stage2_id = summary.loc[summary.stage.eq(2),'loan_id'].iloc[0]
monthly[(monthly.loan_id.eq(stage2_id)) & (monthly.scenario.eq('Base'))][
['future_month','conditional_pd','survival_probability','marginal_pd','lgd','ead','discount_factor','period_ecl']].head(18)

,future_month,conditional_pd,survival_probability,marginal_pd,lgd,ead,discount_factor,period_ecl
0,1,0.0372,1.0000,0.0372,0.0426,"230,343.7877",0.9965,364.6185
1,2,0.0372,0.9508,0.0354,0.0426,"230,067.8214",0.9930,345.0283
2,3,0.0372,0.9041,0.0337,0.0426,"229,790.8778",0.9894,326.4098
3,4,0.0372,0.8596,0.0320,0.0426,"229,512.9534",0.9860,308.7218
4,5,0.0372,0.8173,0.0304,0.0426,"229,234.0446",0.9825,291.9242
5,6,0.0372,0.7772,0.0289,0.0425,"228,954.1480",0.9790,275.9782
6,7,0.0372,0.7389,0.0275,0.0425,"228,673.2602",0.9756,260.8460
7,8,0.0372,0.7026,0.0262,0.0424,"228,391.3775",0.9721,246.4911
8,9,0.0379,0.6680,0.0253,0.0424,"228,108.4965",0.9687,237.1127
9,10,0.0423,0.6348,0.0269,0.0423,"227,824.6136",0.9653,249.9056


In [7]:
stage3_id = summary.loc[summary.stage.eq(3),'loan_id'].iloc[0]
monthly[monthly.loan_id.eq(stage3_id)][
['scenario','ead','projected_property_value','gross_collateral_proceeds','recovery_expenses',
 'expected_mi_recovery','expected_recovery','discount_factor','discounted_expected_recovery','period_ecl']]

,scenario,ead,projected_property_value,gross_collateral_proceeds,recovery_expenses,expected_mi_recovery,expected_recovery,discount_factor,discounted_expected_recovery,period_ecl
1206,Base,"205,696.7900","448,084.3386","394,314.2179","44,808.4339",0.0000,"205,696.7900",0.9369,"192,713.1262","12,983.6638"
1207,Upside,"205,696.7900","461,328.9942","405,969.5149","46,132.8994",0.0000,"205,696.7900",0.9369,"192,713.1262","12,983.6638"
1208,Downside,"205,696.7900","396,529.7873","348,946.2128","39,652.9787",0.0000,"205,696.7900",0.9369,"192,713.1262","12,983.6638"


Stage 3 has no performing-loan marginal PD. Its provision is exposure less discounted scenario-specific expected recovery.